# Análisis de Sentimiento — Harbor House Café

## 1. Objetivos

**Pregunta de negocio:** Determinar si el sentimiento escrito en las 500 reseñas de clientes (`data/raw/reviews.csv`) se alinea con la puntuación promedio del negocio (**4.5 / 5 estrellas**), identificar discrepancias relevantes y dejar lista una ruta de inferencia reproducible.

## 2. Carga del dataset y exploración inicial (EDA)

Cargamos `data/raw/reviews.csv` y revisamos su forma, tipos de datos, valores faltantes y duplicados.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/raw/reviews.csv")
if not data_path.exists():
    data_path = Path("data/raw/reviews.csv")

df = pd.read_csv(data_path)
print(f"Forma del dataset: {df.shape}")
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores faltantes:")
print(df.isna().sum())
print(f"\nFilas duplicadas: {df.duplicated().sum()} | review_id duplicados: {df['review_id'].duplicated().sum()} | review_text duplicados: {df['review_text'].duplicated().sum()}")
df.head()

Forma del dataset: (500, 3)

Tipos de datos:
review_id      int64
rating         int64
review_text      str
dtype: object

Valores faltantes:
review_id      0
rating         0
review_text    0
dtype: int64

Filas duplicadas: 0 | review_id duplicados: 0 | review_text duplicados: 7


### Distribución de ratings y longitud de texto

Calculamos el balance de calificaciones (`rating`), el promedio y las estadísticas de longitud de `review_text`.

In [2]:
rating_counts = df["rating"].value_counts().sort_index()
rating_pct = (df["rating"].value_counts(normalize=True).sort_index() * 100).round(2)
print(pd.DataFrame({"conteo": rating_counts, "porcentaje_%": rating_pct}))
print(f"\nRating promedio real: {df['rating'].mean():.3f}")

texts = df["review_text"].fillna("").astype(str)
df["char_len"] = texts.str.len()
df["word_len"] = texts.str.split().str.len()
print("\nEstadísticas de longitud:")
print(df[["char_len", "word_len"]].describe().round(2))

        conteo  porcentaje_%
rating                      
1           12           2.4
2           18           3.6
3           38           7.6
4           72          14.4
5          360          72.0

Rating promedio real: 4.500

Estadísticas de longitud:
       char_len  word_len
count    500.00    500.00
mean     170.17     29.10
std       27.06      4.82
min       86.00     14.00
25%      155.00     26.00
50%      173.00     29.00
75%      189.00     32.00
max      230.00     42.00


## 3. Insights clave y propuesta de limpieza

### Insights accionables
- **Fuerte desbalance hacia ratings altos:** El 86.4% de las reseñas son de 4 o 5 estrellas (72.0% de 5 y 14.4% de 4), con un promedio exacto de **4.5 / 5**. Las reseñas negativas (1–2 estrellas) representan solo el 6.0%.
- **Textos homogéneos y sin vacíos:** Las 500 reseñas tienen entre 14 y 42 palabras (promedio 29.1 palabras), sin valores nulos ni textos vacíos.
- **Matices mixtos en ratings altos:** En reseñas de 4 y 5 estrellas aparecen quejas puntuales de servicio compensadas por la comida, lo que puede generar falsos negativos al evaluar solo el texto.

### Propuesta de limpieza
- **No hace falta limpieza estructural:** No hay valores nulos, filas ni `review_id` duplicados, ni textos vacíos. Los 7 `review_text` repetidos se conservan para mantener la distribución real.

## 4. Plan de acción y justificación del modelo

- **Modelo fijo:** `nlptown/bert-base-multilingual-uncased-sentiment`, cargado una sola vez con `transformers.pipeline("sentiment-analysis")`.
- **Justificación:** Predice directamente en una escala de 1 a 5 estrellas, alineada con `rating`. Mapeamos las predicciones a bandas de sentimiento: `1–2` → `negative`, `3` → `neutral`, `4–5` → `positive`.
- **Siguiente paso:** Ejecutar inferencia sobre las 500 reseñas, comparar las bandas y el promedio predicho frente a 4.5, analizar falsos negativos e inspeccionar una muestra manual.

## 5. Inferencia sobre las 500 reseñas

Cargamos el modelo una única vez y generamos `pred_rating`, `pred_score` y `sentiment_band`.

In [3]:
from transformers import pipeline

MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
classifier = pipeline("sentiment-analysis", model=MODEL_NAME)

def map_sentiment_band(stars: int) -> str:
    if stars <= 2:
        return "negative"
    if stars == 3:
        return "neutral"
    return "positive"

results = classifier(df["review_text"].tolist(), batch_size=32, truncation=True)
df["pred_rating"] = [int(r["label"].split()[0]) for r in results]
df["pred_score"] = [round(float(r["score"]), 4) for r in results]
df["sentiment_band"] = df["pred_rating"].apply(map_sentiment_band)
df[["review_id", "rating", "pred_rating", "pred_score", "sentiment_band"]].head()

## 6. Desglose de sentimiento vs. promedio de 4.5 estrellas

Comparamos la distribución de bandas (`positive`, `neutral`, `negative`), el promedio predicho y la matriz de confusión frente al `rating` real.

In [4]:
band_pct = (df["sentiment_band"].value_counts(normalize=True) * 100).round(2)
print("Distribución de sentimiento predicho (%):")
print(band_pct)
print(f"\nRating promedio real: {df['rating'].mean():.3f}")
print(f"Rating promedio predicho: {df['pred_rating'].mean():.3f}")
print(f"Exactitud (coincidencia exacta): {(df['rating'] == df['pred_rating']).mean():.3f}")

print("\nMatriz de confusión (Real vs Predicho):")
print(pd.crosstab(df["rating"], df["pred_rating"], rownames=["Real"], colnames=["Pred"]))

Distribución de sentimiento predicho (%):
sentiment_band
positive    82.4
negative     9.4
neutral      8.2
Name: proportion, dtype: float64

Rating promedio real: 4.500
Rating promedio predicho: 4.398
Exactitud (coincidencia exacta): 0.858

Matriz de confusión (Real vs Predicho):
Pred   1  2   3   4    5
Real                    
1      3  9   0   0    0
2      7  9   2   0    0
3      0  3  35   0    0
4      1  0   3  47   21
5     11  4   1   9  335


## 7. Análisis de falsos negativos

Identificamos reseñas donde la calificación humana es alta (`rating >= 4`) pero el modelo predijo `1` o `2` estrellas (`pred_rating <= 2`).

In [5]:
false_negatives = df[(df["rating"] >= 4) & (df["pred_rating"] <= 2)]
print(f"Total de falsos negativos: {len(false_negatives)}")
for _, row in false_negatives.head(8).iterrows():
    print(f"[ID {row['review_id']}] Real: {row['rating']} | Pred: {row['pred_rating']} | {row['review_text']}")

Total de falsos negativos: 16
[ID 14] Real: 4 | Pred: 1 | Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares about the customers. Already planning my next visit.
[ID 15] Real: 5 | Pred: 1 | Every dish was bursting with flavor. The place has such a cozy, welcoming vibe. Already planning my next visit.
[ID 73] Real: 5 | Pred: 1 | Stopped by Harbor House Café for the first time. Service was fast and friendly. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.
[ID 87] Real: 5 | Pred: 1 | Tried Harbor House Café after seeing it recommended online. The seasonal menu never disappoints. They went out of their way to accommodate my allergy. Will definitely be back!
[ID 138] Real: 5 | Pred: 2 | Tried Harbor House Café after seeing it recommended online. They went out of their way to accommodate my allergy. The place has such a cozy, welcoming vibe. Already planning my next visit.
[ID 200] Re

**Hipótesis sobre los falsos negativos:** `nlptown` fue entrenado principalmente con reseñas de productos (e-commerce). En reseñas de servicios/cafetería, expresiones como *"allergy"*, *"recommended online"* o frases mixtas con quejas de espera/personal compensadas por la comida (*"the food made up for it"*) pueden activar señales negativas en el modelo aunque la experiencia global del cliente sea de 4–5 estrellas.

## 8. Muestra manual (18 reseñas)

Inspeccionamos una muestra aleatoria de 18 reseñas para verificar si la etiqueta predicha tiene sentido.

In [6]:
manual_sample = df[["review_id", "rating", "pred_rating", "sentiment_band", "review_text"]].sample(n=18, random_state=42)
for _, row in manual_sample.iterrows():
    match = "OK" if (row["rating"] >= 4 and row["sentiment_band"] == "positive") or (row["rating"] <= 2 and row["sentiment_band"] == "negative") or (row["rating"] == 3 and row["sentiment_band"] == "neutral") else "REVISAR"
    print(f"[{row['review_id']}] {match} | Real={row['rating']} Pred={row['pred_rating']} ({row['sentiment_band']}) | {row['review_text'][:110]}...")

[362] OK | Real=3 Pred=3 (neutral) | Stopped by Harbor House Café for the first time. The coffee was okay. Average service, no complaints. It's fin...
[74] OK | Real=5 Pred=5 (positive) | Stopped by Harbor House Café for the first time. The place has such a cozy, welcoming vibe. Every dish was bur...
[375] OK | Real=5 Pred=5 (positive) | Grabbed a quick coffee at Harbor House Café this morning. Service was fast and friendly. The place has such a ...
[156] OK | Real=4 Pred=4 (positive) | Grabbed a quick coffee at Harbor House Café this morning. The coffee was rich and perfectly brewed. The staff ...
[105] OK | Real=5 Pred=5 (positive) | Grabbed a quick coffee at Harbor House Café this morning. The staff were warm and attentive. The music and dec...
[395] OK | Real=4 Pred=4 (positive) | The food was absolutely delicious. We waited 25 minutes just to get water, but overall a solid experience. Mig...
[378] OK | Real=5 Pred=5 (positive) | Visited Harbor House Café last weekend. Loved the na

## 9. Conclusiones para la Account Manager

- **El sentimiento escrito confirma el 4.5 / 5:** El modelo predice un promedio de **4.398 / 5** con **82.4% de reseñas positivas**, **8.2% neutrales** y **9.4% negativas**, alineado con la calificación pública del negocio.
- **Por qué el texto parece ligeramente más bajo (4.398 vs 4.5):** Existen 16 falsos negativos donde clientes de 4–5 estrellas mencionan detalles de servicio o palabras neutras/mixtas que el modelo de productos penaliza.
- **Siguiente paso en producción:** La lógica limpia queda migrada en `src/app.py`, generando `data/processed/reviews_with_sentiment.csv`.